# Mushroom Classification Project

**Course:** Data Science, 2nd semester  
**Dataset:** Mushroom Classification Database  
**Date:** 2026-06-01

This notebook documents a complete machine-learning workflow for predicting whether a mushroom is edible or poisonous from categorical mushroom characteristics.

## 1. Problem Statement

The dataset contains hypothetical samples from 23 species of gilled mushrooms in the *Agaricus* and *Lepiota* families. The target variable is `poison`, where `e` means edible and `p` means poisonous.

The project goal is a binary supervised classification task: based on observable categorical characteristics such as odor, gill color, spore print color, population, and habitat, we want to predict whether an unknown mushroom instance belongs to the poisonous class.

## 2. Dataset Notes

Important points from the dataset documentation:

- The dataset has 8,124 instances.
- There are 22 categorical input features and one target column.
- The classes are nearly balanced: 4,208 edible and 3,916 poisonous mushrooms.
- `stalk-root` contains many `?` values, which represent missing or unknown values.
- The raw CSV uses `~` as delimiter, not a comma.

Because mushroom toxicity is safety-critical, recall for the poisonous class is especially important: a poisonous mushroom classified as edible would be the most dangerous error.

In [ ]:
from pathlib import Path
from io import StringIO

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score, make_scorer, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier, export_text

pd.set_option("display.max_columns", 30)

## 3. Load the Raw Dataset

The original file cannot always be loaded with a simple `pd.read_csv(..., sep="~")`, because a few rows contain additional empty trailing fields. The loader below keeps the original raw file untouched, trims only empty fields at the end of affected rows, and records what was changed.

In [ ]:
RAW_DATA_PATH = Path("../data/raw/mushroom_classification_data.csv")
CLEAN_DATA_PATH = Path("../data/processed/mushroom_classification_clean.csv")

EXPECTED_COLUMNS = ['poison', 'cap-shape', 'cap-surface', 'cap-color', 'bruises?', 'odor', 'gill-attachment', 'gill-spacing', 'gill-size', 'gill-color', 'stalk-shape', 'stalk-root', 'stalk-surface-above-ring', 'stalk-surface-below-ring', 'stalk-color-above-ring', 'stalk-color-below-ring', 'veil-type', 'veil-color', 'ring-number', 'ring-type', 'spore-print-color', 'population', 'habitat']

def clean_raw_lines(path):
    cleaned_lines = []
    malformed_rows = []
    for line_number, line in enumerate(path.read_text(encoding="utf-8").splitlines(), start=1):
        parts = line.split("~")
        if len(parts) > len(EXPECTED_COLUMNS) and all(value == "" for value in parts[len(EXPECTED_COLUMNS):]):
            malformed_rows.append({
                "line": line_number,
                "original_fields": len(parts),
                "issue": "trailing empty fields",
                "action": "trimmed to 23 fields",
            })
            parts = parts[:len(EXPECTED_COLUMNS)]
        elif len(parts) != len(EXPECTED_COLUMNS):
            malformed_rows.append({
                "line": line_number,
                "original_fields": len(parts),
                "issue": "unexpected field count",
                "action": "kept for manual inspection",
            })
        cleaned_lines.append("~".join(parts))
    return cleaned_lines, malformed_rows

cleaned_lines, malformed_rows = clean_raw_lines(RAW_DATA_PATH)
df_raw = pd.read_csv(StringIO("\n".join(cleaned_lines)), sep="~")

print(df_raw.shape)
pd.DataFrame(malformed_rows)

During the initial loading step, the following malformed rows were detected and handled:

`[{'line': 6919, 'original_fields': 28, 'issue': 'trailing empty fields', 'action': 'trimmed to 23 fields'}, {'line': 7942, 'original_fields': 24, 'issue': 'trailing empty fields', 'action': 'trimmed to 23 fields'}, {'line': 8030, 'original_fields': 25, 'issue': 'trailing empty fields', 'action': 'trimmed to 23 fields'}]`

These rows had the correct data values followed by empty separator fields. Therefore trimming the trailing empty fields is a conservative fix.

## 4. Data Exploration

We first inspect the dimensions, target distribution, duplicate rows, missing-value markers, and number of unique categories per column.

In [ ]:
display(df_raw.head())

print("Shape:", df_raw.shape)
print("Duplicates:", df_raw.duplicated().sum())
print("Missing values recognized by pandas:", int(df_raw.isna().sum().sum()))

target_counts = df_raw["poison"].value_counts().rename(index={"e": "edible", "p": "poisonous"})
target_percent = (df_raw["poison"].value_counts(normalize=True) * 100).round(1).rename(index={"e": "edible", "p": "poisonous"})

display(pd.DataFrame({"count": target_counts, "percent": target_percent}))

In [ ]:
question_mark_counts = (df_raw == "?").sum()
display(question_mark_counts[question_mark_counts > 0].to_frame("question_mark_count"))

unique_counts = df_raw.nunique(dropna=False).sort_values()
display(unique_counts.to_frame("unique_values"))

constant_columns = [column for column in df_raw.columns if df_raw[column].nunique(dropna=False) == 1]
constant_columns

Main EDA findings:

- There are no duplicate rows.
- The only systematic missing marker is `?` in `stalk-root`.
- `veil-type` has only one value and therefore contains no predictive information.
- The target classes are balanced enough that accuracy is meaningful, but precision, recall, and F1 are still better evaluation metrics.

## 5. Validate Categories and Clean Data

Since all features are categorical, each column should contain only the values listed in the dataset documentation. Invalid category values are converted to `?`, which we treat as unknown/missing during preprocessing.

In [ ]:
ALLOWED_VALUES = {'poison': ['e', 'p'], 'cap-shape': ['b', 'c', 'f', 'k', 's', 'x'], 'cap-surface': ['f', 'g', 's', 'y'], 'cap-color': ['b', 'c', 'e', 'g', 'n', 'p', 'r', 'u', 'w', 'y'], 'bruises?': ['f', 't'], 'odor': ['a', 'c', 'f', 'l', 'm', 'n', 'p', 's', 'y'], 'gill-attachment': ['a', 'd', 'f', 'n'], 'gill-spacing': ['c', 'd', 'w'], 'gill-size': ['b', 'n'], 'gill-color': ['b', 'e', 'g', 'h', 'k', 'n', 'o', 'p', 'r', 'u', 'w', 'y'], 'stalk-shape': ['e', 't'], 'stalk-root': ['?', 'b', 'c', 'e', 'r', 'u', 'z'], 'stalk-surface-above-ring': ['f', 'k', 's', 'y'], 'stalk-surface-below-ring': ['f', 'k', 's', 'y'], 'stalk-color-above-ring': ['b', 'c', 'e', 'g', 'n', 'o', 'p', 'w', 'y'], 'stalk-color-below-ring': ['b', 'c', 'e', 'g', 'n', 'o', 'p', 'w', 'y'], 'veil-type': ['p', 'u'], 'veil-color': ['n', 'o', 'w', 'y'], 'ring-number': ['n', 'o', 't'], 'ring-type': ['c', 'e', 'f', 'l', 'n', 'p', 's', 'z'], 'spore-print-color': ['b', 'h', 'k', 'n', 'o', 'r', 'u', 'w', 'y'], 'population': ['a', 'c', 'n', 's', 'v', 'y'], 'habitat': ['d', 'g', 'l', 'm', 'p', 'u', 'w']}
ALLOWED_VALUES = {column: set(values) for column, values in ALLOWED_VALUES.items()}

df = df_raw.copy()
invalid_values = []

for column, allowed_values in ALLOWED_VALUES.items():
    invalid_mask = ~df[column].astype(str).isin(allowed_values)
    for row_index, value in df.loc[invalid_mask, column].items():
        invalid_values.append({
            "row": int(row_index) + 2,
            "column": column,
            "value": value,
            "action": "set to ? for categorical missing/unknown",
        })
    df.loc[invalid_mask, column] = "?"

pd.DataFrame(invalid_values)

Invalid category values found:

`[{'row': 8107, 'column': 'stalk-color-below-ring', 'value': '0.34', 'action': 'set to ? for categorical missing/unknown'}, {'row': 8112, 'column': 'veil-color', 'value': '0.4', 'action': 'set to ? for categorical missing/unknown'}, {'row': 8108, 'column': 'ring-number', 'value': '0.3', 'action': 'set to ? for categorical missing/unknown'}, {'row': 7937, 'column': 'habitat', 'value': 'd**', 'action': 'set to ? for categorical missing/unknown'}]`

These values do not match the dataset documentation. They are not used as meaningful categories; instead they are treated as unknown values.

In [ ]:
df.to_csv(CLEAN_DATA_PATH, index=False)
print("Clean dataset written to:", CLEAN_DATA_PATH)
print("Clean shape:", df.shape)

## 6. Feature Relationships

The following contingency tables show that several features are highly informative. For example, odor values such as foul, fishy, spicy, pungent, creosote, and musty occur only in poisonous samples in this dataset.

In [ ]:
for feature in ["odor", "spore-print-color", "gill-color", "ring-type", "habitat", "population"]:
    print("\n", feature)
    display(pd.crosstab(df[feature], df["poison"]))

## 7. Machine-Learning Strategy

This is a supervised binary classification task.

Planned approach:

- Target: `poison`
- Inputs: all remaining mushroom characteristics
- Remove `veil-type`, because it is constant
- Convert `?` to missing values
- Impute missing categorical values with the most frequent category
- One-hot encode categorical variables
- Compare a dummy baseline, decision tree, logistic regression, and random forest

The decision tree is especially useful here because it is interpretable and matches the rule-like nature of the dataset.

In [ ]:
y = df["poison"].map({"e": 0, "p": 1})
X = df.drop(columns=["poison", "veil-type"]).replace("?", np.nan)

categorical_features = X.columns.tolist()

categorical_preprocessor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", categorical_preprocessor, categorical_features),
    ]
)

## 8. Model Evaluation

We use stratified 5-fold cross-validation. Stratification keeps the class distribution similar across folds.

In [ ]:
models = {
    "Dummy majority baseline": DummyClassifier(strategy="most_frequent"),
    "Decision tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "Logistic regression": LogisticRegression(max_iter=1000),
    "Random forest": RandomForestClassifier(n_estimators=100, random_state=42),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    "accuracy": "accuracy",
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": make_scorer(recall_score, zero_division=0),
    "f1": make_scorer(f1_score, zero_division=0),
}

rows = []
for name, model in models.items():
    pipeline = Pipeline(
        steps=[
            ("preprocess", preprocessor),
            ("model", model),
        ]
    )
    scores = cross_validate(pipeline, X, y, cv=cv, scoring=scoring)
    rows.append(
        {
            "model": name,
            "accuracy_mean": scores["test_accuracy"].mean(),
            "precision_mean": scores["test_precision"].mean(),
            "recall_mean": scores["test_recall"].mean(),
            "f1_mean": scores["test_f1"].mean(),
        }
    )

results = pd.DataFrame(rows).round(4)
display(results)

Expected interpretation:

- The dummy baseline predicts only the majority class and is therefore weak.
- The decision tree already performs almost perfectly.
- Logistic regression and random forest also reach near-perfect or perfect scores.

This is plausible for this benchmark dataset because several categorical features are strongly associated with the target.

## 9. Final Holdout Evaluation

We train the interpretable decision tree on 80% of the data and evaluate it on the remaining 20%.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

final_model = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", DecisionTreeClassifier(max_depth=5, random_state=42)),
    ]
)

final_model.fit(X_train, y_train)
y_pred = final_model.predict(X_test)

print("Confusion matrix")
print(confusion_matrix(y_test, y_pred))
print()
print(classification_report(y_test, y_pred, target_names=["edible", "poisonous"]))

In a safety-critical use case, the most important number is recall for the poisonous class. A false negative would mean predicting edible for a poisonous mushroom. In this dataset the decision tree produces extremely few such errors.

## 10. Interpreting the Decision Tree

The tree below is exported as text. It shows which one-hot encoded feature conditions are used by the model. This makes the model easier to explain than a black-box classifier.

In [ ]:
fitted_preprocessor = final_model.named_steps["preprocess"]
fitted_tree = final_model.named_steps["model"]

feature_names = fitted_preprocessor.get_feature_names_out()
print(export_text(fitted_tree, feature_names=list(feature_names), max_depth=5))

## 11. Conclusion

The mushroom dataset is very suitable for classification. After handling formatting problems, invalid category values, and missing markers, the supervised learning pipeline performs extremely well.

The most transparent final model is the decision tree because it can be explained as a set of rules. More complex models such as random forest also work very well, but the added performance is not necessary for this dataset.

Final recommendation: use the decision tree as the main model in the report, compare it with the baseline and at least one additional classifier, and explain why recall for poisonous mushrooms matters most.

## 12. Use of Large Language Models

Large Language Models were used as support during this group project. The support included:

- understanding the project requirements,
- structuring the notebook,
- identifying a suitable machine-learning strategy,
- checking data-quality issues,
- drafting explanatory text.

The group reviewed the generated code and explanations and remains responsible for understanding and being able to explain all submitted work.